In [2]:
# Install Google's Gemini SDK
%pip install -q -U google-genai

# Import required libraries
from google import genai
from kaggle_secrets import UserSecretsClient

# Access your Kaggle secret
secrets = UserSecretsClient()
api_key = secrets.get_secret("DR Report")

# Create Gemini client
client = genai.Client(api_key=api_key)

print("Gemini API connection configured successfully!")

Note: you may need to restart the kernel to use updated packages.
Gemini API connection configured successfully!


In [3]:
# Select the Gemini model
MODEL_NAME = "gemini-3.6-flash"

# Send a test prompt to Gemini
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain what an LLM is in simple words in 3 sentences."
)

# Display the generated response
print(response.text)

A Large Language Model (LLM) is a type of artificial intelligence that has read massive amounts of text to learn how humans communicate. It works by recognizing patterns in words, allowing it to answer questions, write essays, and hold realistic conversations. You can think of it as a super-smart digital assistant that powers modern AI tools like ChatGPT.


In [4]:
# Sample output from our CNN model
sample_prediction = {
    "predicted_class": "Moderate",
    "confidence": 0.76,
    "probabilities": {
        "No DR": 0.04,
        "Mild": 0.10,
        "Moderate": 0.76,
        "Severe": 0.07,
        "Proliferative DR": 0.03
    }
}

print(sample_prediction)

{'predicted_class': 'Moderate', 'confidence': 0.76, 'probabilities': {'No DR': 0.04, 'Mild': 0.1, 'Moderate': 0.76, 'Severe': 0.07, 'Proliferative DR': 0.03}}


In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".keras"):
            print(os.path.join(root, file))

/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras


In [3]:
import tensorflow as tf

MODEL_PATH = "/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras"

cnn_model = tf.keras.models.load_model(MODEL_PATH)

print("CNN model loaded successfully!")
cnn_model.summary()

CNN model loaded successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,574,545 (204.37 MB)

 Trainable params: 14,988,293 (57.18 MB)

 Non-trainable params: 8,609,664 (32.84 MB)

 Optimizer params: 29,976,588 (114.35 MB)

In [4]:
import os

DATASET_PATH = "/kaggle/input/competitions/aptos2019-blindness-detection"

print(os.listdir(DATASET_PATH))

['sample_submission.csv', 'train_images', 'train.csv', 'test.csv', 'test_images']


In [5]:
import pandas as pd
import os

# Load the dataset labels
csv_path = os.path.join(DATASET_PATH, "train.csv")
df = pd.read_csv(csv_path)

# Select the first retinal image
sample_image_name = df.iloc[0]["id_code"]
actual_label = df.iloc[0]["diagnosis"]

# Get its image path
image_path = os.path.join(
    DATASET_PATH,
    "train_images",
    sample_image_name + ".png"
)

print("Image name:", sample_image_name)
print("Actual DR grade:", actual_label)
print("Image path:", image_path)

Image name: 000c1434d8d7
Actual DR grade: 2
Image path: /kaggle/input/competitions/aptos2019-blindness-detection/train_images/000c1434d8d7.png


In [6]:
import tensorflow as tf

# Read the retinal image
image = tf.io.read_file(image_path)

# Decode the image into RGB format
image = tf.image.decode_png(image, channels=3)

# Resize to the model's expected input size
image = tf.image.resize(image, [224, 224])

# Apply ResNet50 preprocessing
image = tf.keras.applications.resnet50.preprocess_input(image)

# Add batch dimension
image = tf.expand_dims(image, axis=0)

print("Prepared image shape:", image.shape)
print("Image preprocessing completed!")

Prepared image shape: (1, 224, 224, 3)
Image preprocessing completed!


In [7]:
# Get predictions from the CNN
predictions = cnn_model.predict(image)

# Get the class with the highest probability
predicted_class = predictions.argmax(axis=1)[0]

# Get the confidence score
confidence = predictions[0][predicted_class]

# DR class names
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Predicted DR Grade:", class_names[predicted_class])
print("Confidence:", round(float(confidence) * 100, 2), "%")

print("\nClass-wise probabilities:")

for i, probability in enumerate(predictions[0]):
    print(class_names[i], ":", round(float(probability) * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Predicted DR Grade: Moderate
Confidence: 69.47 %

Class-wise probabilities:
No DR : 0.02 %
Mild : 2.55 %
Moderate : 69.47 %
Severe : 4.39 %
Proliferative DR : 23.56 %


In [8]:
print("Actual DR Grade:", class_names[int(actual_label)])
print("Predicted DR Grade:", class_names[predicted_class])

if int(actual_label) == predicted_class:
    print("Result: Correct Prediction")
else:
    print("Result: Incorrect Prediction")

Actual DR Grade: Moderate
Predicted DR Grade: Moderate
Result: Correct Prediction


In [9]:
# Convert CNN prediction into structured data
cnn_output = {
    "predicted_class": class_names[predicted_class],
    "confidence": float(confidence),
    "probabilities": {
        class_names[i]: float(predictions[0][i])
        for i in range(len(class_names))
    }
}

print(cnn_output)

{'predicted_class': 'Moderate', 'confidence': 0.6947429180145264, 'probabilities': {'No DR': 0.00021847766765858978, 'Mild': 0.025516953319311142, 'Moderate': 0.6947429180145264, 'Severe': 0.04388175532221794, 'Proliferative DR': 0.23563992977142334}}


In [12]:
from google import genai
from kaggle_secrets import UserSecretsClient

# Retrieve your Gemini API key
secrets = UserSecretsClient()
api_key = secrets.get_secret("DR Report")

# Create Gemini client
client = genai.Client(api_key=api_key)

# Select Gemini model
MODEL_NAME = "gemini-3.6-flash"

print("Gemini connected successfully!")

Gemini connected successfully!


In [13]:
# Create a prompt using the actual CNN prediction
prompt = f"""
You are an AI-assisted report drafting assistant
for a diabetic retinopathy screening research prototype.

Here is the prediction generated by our CNN model:

{cnn_output}

Generate a structured screening report with the following sections:

1. Predicted DR Grade
2. Model Confidence
3. Class-wise Probability Analysis
4. Interpretation of the Model Prediction
5. Model Uncertainty
6. Limitations
7. Recommendation for Professional Review

Instructions:
- Report this as a model prediction, not a confirmed diagnosis.
- Explain the prediction in simple language.
- Use only the information provided by the CNN.
- Do not invent retinal findings, symptoms, or patient history.
- Do not prescribe medicines or suggest treatments.
- Mention that this is an experimental research prototype.
- Recommend review by a qualified eye-care professional.
"""

# Send the prompt to Gemini
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt
)

# Display the generated report
print(response.text)

### Automated AI Screening Report (Research Prototype)

---

### 1. Predicted DR Grade
* **Predicted Class:** Moderate Diabetic Retinopathy (Moderate DR)

---

### 2. Model Confidence
* **Primary Confidence:** 69.47% (0.6947)

---

### 3. Class-wise Probability Analysis
Below is the full probability distribution calculated by the CNN model across all screening categories:

* **No DR:** 0.02% (0.0002)
* **Mild DR:** 2.55% (0.0255)
* **Moderate DR:** 69.47% (0.6947)
* **Severe DR:** 4.39% (0.0439)
* **Proliferative DR:** 23.56% (0.2356)

---

### 4. Interpretation of the Model Prediction
The AI model analyzed the image and assigned the highest probability (69.47%) to **Moderate Diabetic Retinopathy**. In simple terms, the algorithm determined that the visual patterns in the input image align most closely with moderate signs of diabetic retinopathy compared to the other classes. 

Additionally, the model assigned a secondary probability of 23.56% to **Proliferative DR**, indicating that w

In [14]:

prompt_v2 = f"""
You are an AI-assisted medical report writing assistant
supporting an experimental Diabetic Retinopathy (DR)
screening research project.

Your task is to explain the output of a CNN classification
model in a clear, professional and structured format.

IMPORTANT RULES:
1. You are given only the CNN's classification results.
2. You have NOT examined the retinal image.
3. Do not claim to have detected lesions, hemorrhages,
   exudates, microaneurysms or other visual features.
4. Do not invent patient details, symptoms or medical history.
5. Do not present the prediction as a confirmed diagnosis.
6. Explain uncertainty and the limitations of the model.
7. Recommend review by a qualified eye-care professional.
8. Do not prescribe medication or treatment.

CNN MODEL OUTPUT:
{cnn_output}

Generate a professional report using these sections:

1. REPORT TITLE
   Diabetic Retinopathy AI-Assisted Screening Report

2. PREDICTED DR GRADE
   State the predicted grade provided by the CNN.

3. MODEL PREDICTION PROBABILITY
   State the top-class probability as a model score.
   Do not call it a clinically validated confidence level.

4. CLASS-WISE PROBABILITY ANALYSIS
   Present all five DR grades and their probabilities.
   Explain which class received the highest probability.

5. INTERPRETATION
   Explain what the model's predicted grade represents.
   Use only the supplied classification output.

6. UNCERTAINTY
   Discuss the possibility of misclassification.
   Mention other classes with notable probabilities.

7. MODEL LIMITATIONS
   Explain that this is an experimental AI model and
   its prediction is not a confirmed medical diagnosis.

8. PROFESSIONAL REVIEW
   State that the result must be reviewed by a qualified
   ophthalmologist or eye-care professional.

FORMATTING:
- Use clear headings.
- Use professional medical-report language.
- Use short, meaningful paragraphs.
- Use percentages rounded to two decimal places.
- Do not add information that is not provided.
- Return only the report.
"""

print("Prompt Version 2 created successfully!")


Prompt Version 2 created successfully!


In [15]:

response_v2 = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt_v2
)

report_v2 = response_v2.text

print(report_v2)


### 1. REPORT TITLE
Diabetic Retinopathy AI-Assisted Screening Report

### 2. PREDICTED DR GRADE
Predicted DR Grade: Moderate

### 3. MODEL PREDICTION PROBABILITY
Top-Class Model Score: 69.47%

This score represents the statistical output of the classification algorithm for the highest-ranked class and should not be interpreted as a clinically validated confidence level.

### 4. CLASS-WISE PROBABILITY ANALYSIS
The model evaluated the input across five Diabetic Retinopathy (DR) categories with the following probability distribution:

* **No DR:** 0.02%
* **Mild:** 2.55%
* **Moderate:** 69.47%
* **Severe:** 4.39%
* **Proliferative DR:** 23.56%

The "Moderate" category received the highest probability score from the model at 69.47%.

### 5. INTERPRETATION
The CNN model's output indicates a primary classification of Moderate Diabetic Retinopathy based strictly on the processed algorithm features. This finding represents only the mathematical output of the deep learning model and does not r